In [25]:
import os 
import torch 
import torch.nn as nn
import torchvision.models  as models 
import torchvision.transforms as transforms 
import torchvision.transforms.functional as TF
import glob
from PIL import Image 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-->Execution target confirmed: {device}")
if torch.cuda.is_available(): 
    print(f"--> Cloud Hardware Profile: {torch.cuda.get_device_name(0)}") 
    
backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

modules = list(backbone.children())[:-2]
feature_extractor = nn.Sequential(*modules).to(device) 
feature_extractor.eval() 

print("--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully")

-->Execution target confirmed: cuda
--> Cloud Hardware Profile: Tesla T4
--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully


In [26]:
import torch.nn as nn 
import torch.nn.functional as F 

class SaliencyDecoder(nn.Module): 
    def __init__(self): 
        super(SaliencyDecoder, self).__init__() 
        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 1, kernel_size=1)
        
    def forward(self, x): 
        x = F.relu(self.conv1(x)) 
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False) 
        x = F.relu(self.conv3(x))
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False) 
        saliency_map = torch.sigmoid(x)
        return saliency_map

decoder = SaliencyDecoder().to(device)

In [27]:
from torch.utils.data import Dataset, DataLoader 

class SynthethicSaliencyDataset(Dataset): 
    def __init__(self, num_samples=16): 
        self.num_samples = num_samples 
    
    def __len__(self): 
        return self.num_samples 
    
    def __getitem__(self, idx): 
        synthethic_image = torch.rand(3, 224, 224) 
        synthethic_fixation = torch.rand(1, 224, 224) 
        
        return synthethic_image, synthethic_fixation

test_dataset = SynthethicSaliencyDataset(num_samples=8) 
test_loader = DataLoader(test_dataset, batch_size = 2, shuffle = True)

In [28]:
import torch 

def compute_covariance_manifold(features): 
    B, C, H, W = features.shape 
    N = H * W
    
    reshaped_feats = features.view(B, C, N) 
    mean_feats = reshaped_feats.mean(dim=2, keepdim=True)
    centered_feats = reshaped_feats - mean_feats
    cov_matrices = torch.bmm(centered_feats, centered_feats.transpose(1, 2)) / (N - 1)
    eps = 1e-6 * torch.eye(C, device=features.device).unsqueeze(0).repeat(B, 1, 1)
    spd_matrices = cov_matrices + eps 
    
    return spd_matrices

In [29]:
def matrix_logarithm(spd_matrix): 
    eigenvalues, eigenvectors = torch.linalg.eigh(spd_matrix)
    clipped_evals = torch.clamp(eigenvalues, min=1e-6)
    log_evals = torch.log(clipped_evals) 
    log_diagonal = torch.diag_embed(log_evals)
    matrix_log = torch.bmm(torch.bmm(eigenvectors, log_diagonal), eigenvectors.transpose(1, 2))
    
    return matrix_log

def riemannian_smoothness_loss(features): 
    spd_mats = compute_covariance_manifold(features)
    tangent_projections = matrix_logarithm(spd_mats) 
    batch_mean = tangent_projections.mean(dim=0, keepdim=True) 
    roughness_score = torch.mean((tangent_projections - batch_mean) ** 2)
    
    return roughness_score

In [30]:
def compute_cc(pred_map, gt_map): 
     B, C, H, W = pred_map.shape 
     p = pred_map.view(B, -1) 
     g = gt_map.view(B, -1) 
     
     p_mean = p.mean(dim=1, keepdim=True)
     g_mean = g.mean(dim=1, keepdim=True)
     p_centered = p - p_mean 
     g_centered = g - g_mean 
     
     p_norm = torch.norm(p_centered, p=2, dim=1, keepdim=True)
     g_norm = torch.norm(g_centered, p=2, dim=1, keepdim=True) 
     
     eps = 1e-6 
     cc = torch.sum(p_centered * g_centered, dim=1, keepdim=True) / (p_norm * g_norm + eps)
     
     return cc.mean() 

def compute_sim(pred_map, gt_map): 
    B, C, H, W = pred_map.shape 
    p = pred_map.view(B, -1) 
    g = gt_map.view(B, -1) 
    eps = 1e-6 
    
    p_pdf = p / (p.sum(dim=1, keepdim=True) + eps)
    g_pdf = g / (g.sum(dim=1, keepdim=True)+ eps)
    
    sim = torch.sum(torch.minimum(p_pdf, g_pdf), dim=1)
    
    return sim.mean() 

In [31]:
print("--> Commencing optimized pipeline verification lopp....\n")

images, fixation_maps = next(iter(test_loader))  
batch_size = images.shape[0] 
fixation_maps = fixation_maps.to(device)
print(f"[Data Input] Batch Size detected: {batch_size} | Ground Truth Fixations Shape={fixation_maps.shape}")

mock_features = torch.rand(batch_size, 2048, 7, 7, device=device)
print(f"[Backbone] Simulated Deep Feature Map Shape: {mock_features.shape}")

mock_predictions = torch.rand(batch_size, 1, 224, 224, device=device)
print(f"[Decooder] Simulated Predicted Saliency Shape: {mock_predictions.shape}")

try: 
    smoothness_penalty = riemannian_smoothness_loss(mock_features)
    batch_cc = compute_cc(mock_predictions, fixation_maps)
    batch_sim = compute_sim(mock_predictions, fixation_maps) 
    print("\n" + "="*50)
    print(" METRIC VERIFICATION RESULTS")
    print("="*50)
    print(f" Riemannian Smoothness Loss (Batch Variance): {smoothness_penalty.item():.6f}")
    print(f" Saliency Correlation Coefficient (CC):     {batch_cc.item():.6f}")
    print(f" Saliency Similarity Distribution (SIM):    {batch_sim.item():.6f}")
    print("="*50)
    print("\n--> Verification complete! Complete tensor chain is mathematically sound.")
    
except Exception as e: 
    print(f"\n[Error] Pipeline verification failed. Stack trace:\n{e}")

--> Commencing optimized pipeline verification lopp....

[Data Input] Batch Size detected: 2 | Ground Truth Fixations Shape=torch.Size([2, 1, 224, 224])
[Backbone] Simulated Deep Feature Map Shape: torch.Size([2, 2048, 7, 7])
[Decooder] Simulated Predicted Saliency Shape: torch.Size([2, 1, 224, 224])

 METRIC VERIFICATION RESULTS
 Riemannian Smoothness Loss (Batch Variance): 0.001266
 Saliency Correlation Coefficient (CC):     -0.004463
 Saliency Similarity Distribution (SIM):    0.666181

--> Verification complete! Complete tensor chain is mathematically sound.


In [32]:
import random
import os 
import glob 
import torch
from PIL import Image 
from torch.utils.data import Dataset, DataLoader 
import torchvision.transforms.functional as TF


class CAT2000Dataset(Dataset): 
    OFFICIAL_CATEGORIES = [
        "Action", "Affective", "Art", "Cartoon", "DigitalArt", 
        "Everyday", "Fractal", "Indoor", "Inverted", "LightNormal", 
        "LineDrawing", "Loud", "LowResolution", "Natural", "Object", 
        "Outdoor", "Photo", "Random", "Social", "Synthetic"
    ]
    
    def __init__(self, root_dir, split='train', transform=True):
        self.root_dir = root_dir 
        self.split = split 
        self.transform = transform  
        
        self.stimuli_dir = os.path.join(root_dir, "Stimuli")
        self.maps_dir = os.path.join(root_dir, "FixationMaps")
        self.cat_to_idx = {cat: idx for idx, cat in enumerate(self.OFFICIAL_CATEGORIES)} 
        
        self.samples = [] 
        self._index_dataset() 
        
    def _index_dataset(self): 
        for cat in self.OFFICIAL_CATEGORIES: 
            cat_stim_path = os.path.join(self.stimuli_dir, cat)
            cat_maps_path = os.path.join(self.maps_dir, cat)
            
            if os.path.exists(cat_stim_path) and os.path.exists(cat_maps_path): 
                # Grab files sitting ONLY directly inside the category folder
                search_pattern = os.path.join(cat_stim_path, "*.jpg")
                images = sorted(glob.glob(search_pattern))
                
                for img_path in images: 
                    base_name = os.path.basename(img_path)
                    
                    # Explicitly bypass if glob accidentally hits a directory named 'Output.jpg'
                    if os.path.isdir(img_path):
                        continue
                        
                    # Target map directly maps to trainSet/FixationMaps/Category/001.jpg
                    map_path = os.path.join(cat_maps_path, base_name)
                    
                    if os.path.exists(map_path): 
                        self.samples.append({ 
                            "image": img_path, 
                            "map": map_path, 
                            "category_name": cat,
                            "category_idx": self.cat_to_idx[cat]
                        })
                        
        if len(self.samples) == 0: 
            print(f"[STATUS] Target folders initialized at '{self.root_dir}'.")
            print("         Awaiting production dataset upload into category subdirectories.")
        else: 
            print(f"--> Successfully indexed {len(self.samples)} aligned CAT2000 samples across the 20 categories.")
            
    def __len__(self): 
        return len(self.samples)
    
    def __getitem__(self, idx): 
        sample = self.samples[idx]
        
        image = Image.open(sample["image"]).convert("RGB")
        fix_map = Image.open(sample["map"]).convert("L")
        target_size = (224, 224) 
        
        image  = TF.resize(image, target_size, interpolation=TF.InterpolationMode.BILINEAR)
        fix_map = TF.resize(fix_map, target_size, interpolation=TF.InterpolationMode.BILINEAR)
        
        if self.transform and self.split == "train": 
            if torch.rand(1).item() > 0.5: 
                image = TF.hflip(image) 
                fix_map = TF.hflip(fix_map)
                
            if torch.rand(1).item() > 0.5: 
                brightness_val = random.uniform(0.8, 1.2)
                contrast_val = random.uniform(0.8, 1.2)
                image = TF.adjust_brightness(image, brightness_factor=brightness_val)
                image = TF.adjust_contrast(image, contrast_factor=contrast_val)
                
        image_tensor = TF.to_tensor(image)
        fix_tensor = TF.to_tensor(fix_map)
        
        if fix_tensor.max() > 0: 
            fix_tensor = (fix_tensor - fix_tensor.min()) / (fix_tensor.max() - fix_tensor.min() + 1e-6) 
            
        image_tensor = TF.normalize(image_tensor, mean=[0.485, 0.456, 0.406], 
                                    std = [0.229, 0.224, 0.225])
        
        return image_tensor, fix_tensor, sample["category_idx"]
    
    
base_path = "trainSet"
cat2000_production_set  = CAT2000Dataset(root_dir=base_path, split='train', transform=True)
production_loader = DataLoader(cat2000_production_set, batch_size=2, shuffle=True)

if len(cat2000_production_set) > 0:
    imgs, maps, cat_ids = next(iter(production_loader))
    print("\n" + "="*50)
    print(" PRODUCTION DATA INGESTION TESTING")
    print("="*50)
    print(f" Normalized Input Image Batch Shape: {imgs.shape}")
    print(f" Aligned Target Fixation Batch Shape: {maps.shape}")
    print(f" Tracked Category Categorical IDs:  {cat_ids.tolist()}")
    print("="*50)
    print("--> Cell 8 pipeline fully validated! Ready to accept full scale data streams.")

[STATUS] Target folders initialized at 'trainSet'.
         Awaiting production dataset upload into category subdirectories.


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
import torch.optim as optim 

decoder = decoder.to(device)
feature_extractor = feature_extractor.to(device)

optimizer = optim.Adam(decoder.parameters(), lr=1e-4)
NUM_EPOCHS= 3
ALPHA = 0.05 

for epoch in range(NUM_EPOCHS): 
    decoder.train()
    feature_extractor.eval()
    
    epoch_task_loss = 0.0 
    epoch_smooth_loss = 0.0 
    epoch_total_loss = 0.0 
    
    for batch_idx, (images, fixation_maps, category_ids) in enumerate(production_loader): 
        optimizer.zero_grad() 
        images = images.to(device)
        fixation_maps = fixation_maps.to(device)
        
        with torch.no_grad(): 
            raw_features = feature_extractor(images)
            
        raw_features = raw_features.detach().requires_grad_(True)
        predicted_saliency = decoder(raw_features)
        cc_score = compute_cc(predicted_saliency, fixation_maps) 
        task_loss = 1 - cc_score 
        
        smoothness_loss = riemannian_smoothness_loss(raw_features)
        total_loss = task_loss + (ALPHA * smoothness_loss) 
        
        total_loss.backward() 
        optimizer.step() 
        
        epoch_task_loss += task_loss.item() 
        epoch_smooth_loss += smoothness_loss.item() 
        epoch_total_loss += total_loss.item() 
    
    num_batches = len(production_loader) 
    avg_task = epoch_task_loss / num_batches 
    avg_smooth = epoch_smooth_loss / num_batches 
    avg_total = epoch_total_loss / num_batches
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"  -> Task Loss (1 - CC):         {avg_task:.6f}")
    print(f"  -> Riemannian Smoothness Loss: {avg_smooth:.6f}")
    print(f"  -> Total Combined Loss:        {avg_total:.6f}\n")
    
print("="*50)
print("--> Training loop execution successfully verified!")
print("--> Model weights updated and ready for real data streaming.")
print("="*50)